# Europa Colab training driver

Set Colab to **Runtime → Change runtime type → GPU** first. Then edit the variables in the next cell and run downward.

This is intentionally a thin driver notebook: clone/sync repo, generate data if needed, write the TOML config, train, evaluate.

In [1]:
# === EDIT THESE ===
REPO_URL = "https://github.com/alvertremantel/europa.git"
BRANCH = "dev"
PROJECT_DIR = "/content/europa"

# Keep DATA_DIR/RUN_DIR on Drive so generated data and checkpoints survive Colab resets.
USE_GOOGLE_DRIVE = True
DRIVE_ROOT = "/content/drive/MyDrive/europa-colab"
DATA_DIR = f"{DRIVE_ROOT}/data/type-place"
RUN_DIR = f"{DRIVE_ROOT}/runs/e4small-ls"
CONFIG_PATH = f"{DRIVE_ROOT}/train-config.toml"

# Dataset generation. Set GENERATE_DATASET=False if DATA_DIR already has train/val/test/meta.toml.
GENERATE_DATASET = False
FORCE_REGENERATE_DATASET = False
DATASET_SEED = 42

# Runtime/resume. For continuing an interrupted run, keep AUTO_RESUME=True and set ADDITIONAL_EPOCHS.
DEVICE = "cuda"
SEED = 42
AUTO_RESUME = False
RESUME_FROM = ""  # explicit checkpoint path, or blank to disable
ADDITIONAL_EPOCHS = None  # e.g. 2 to add two epochs when resuming

# Model shape. d_model must be divisible by n_heads.
SEQUENCE_LENGTH = 64
D_MODEL = 12
N_HEADS = 3
N_LAYERS = 6
MLP_HIDDEN = 48
DROPOUT = 0.1
POSITION_ENCODING = "type_place"

# Optimization/logging. Raise/lower BATCH_SIZE based on GPU memory.
BATCH_SIZE = 512
EPOCHS = 50
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0
LOG_INTERVAL = 100
EVAL_BATCHES = 25
EXACT_MATCH_SAMPLES = 128
MAX_NEW_TOKENS = 24


In [2]:
from pathlib import Path
import shutil
import subprocess
import sys

def run(cmd, cwd=None):
    cmd = [str(part) for part in cmd]
    print("\n$", " ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True)

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)

run([sys.executable, "-m", "pip", "install", "-q", "uv"])

project = Path(PROJECT_DIR)
if not (project / ".git").exists():
    if project.exists():
        shutil.rmtree(project)
    run(["git", "clone", "--branch", BRANCH, REPO_URL, project])
else:
    run(["git", "fetch", "origin"], cwd=project)
    run(["git", "checkout", BRANCH], cwd=project)
    run(["git", "pull", "--ff-only"], cwd=project)

# The project requires Python 3.12. uv can install/use it even if the Colab kernel differs.
run(["uv", "python", "pin", "3.12"], cwd=project)
run(["uv", "sync", "--python", "3.12"], cwd=project)
run(["uv", "run", "python", "-c", "import sys, torch; print(sys.version); print('torch', torch.__version__, 'cuda?', torch.cuda.is_available(), 'torch cuda', torch.version.cuda); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO CUDA DEVICE')"], cwd=project)


Mounted at /content/drive

$ /usr/bin/python3 -m pip install -q uv

$ git clone --branch dev https://github.com/alvertremantel/europa.git /content/europa

$ uv python pin 3.12

$ uv sync --python 3.12

$ uv run python -c import sys, torch; print(sys.version); print('torch', torch.__version__, 'cuda?', torch.cuda.is_available(), 'torch cuda', torch.version.cuda); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO CUDA DEVICE')


In [3]:
data_path = Path(DATA_DIR)
if GENERATE_DATASET:
    if FORCE_REGENERATE_DATASET and data_path.exists():
        shutil.rmtree(data_path)
    if (data_path / "train.txt").exists() and (data_path / "meta.toml").exists():
        print(f"Dataset already exists at {data_path}; skipping generation.")
    else:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        run(["uv", "run", "generate", "--output-dir", data_path, "--seed", DATASET_SEED], cwd=project)
else:
    print(f"Using existing dataset at {data_path}")



$ uv run generate --output-dir /content/drive/MyDrive/europa-colab/data/type-place --seed 42


In [4]:
import json

assert D_MODEL % N_HEADS == 0, "D_MODEL must be divisible by N_HEADS"
assert POSITION_ENCODING == "type_place", "Only type_place is supported for fresh configs"

def toml_str(value):
    return json.dumps(str(value))

def toml_bool(value):
    return "true" if bool(value) else "false"

def toml_optional_int(value):
    return '""' if value is None else str(int(value))

config_text = f'''# Generated by notebooks/colab-training.ipynb

[paths]
data_dir = {toml_str(DATA_DIR)}
output_dir = {toml_str(RUN_DIR)}

[runtime]
device = {toml_str(DEVICE)}
seed = {SEED}

[resume]
resume_from = {toml_str(RESUME_FROM)}
auto_resume = {toml_bool(AUTO_RESUME)}
additional_epochs = {toml_optional_int(ADDITIONAL_EPOCHS)}

[model]
sequence_length = {SEQUENCE_LENGTH}
d_model = {D_MODEL}
n_heads = {N_HEADS}
n_layers = {N_LAYERS}
mlp_hidden = {MLP_HIDDEN}
dropout = {DROPOUT}
position_encoding = {toml_str(POSITION_ENCODING)}

[optimization]
batch_size = {BATCH_SIZE}
epochs = {EPOCHS}
learning_rate = {LEARNING_RATE}
weight_decay = {WEIGHT_DECAY}
grad_clip = {GRAD_CLIP}

[logging]
log_interval = {LOG_INTERVAL}
eval_batches = {EVAL_BATCHES}
exact_match_samples = {EXACT_MATCH_SAMPLES}
max_new_tokens = {MAX_NEW_TOKENS}

[checkpoint]
checkpoint_keep_last = 5
checkpoint_max_kept = 10
checkpoint_keep_best = 1
checkpoint_jump_threshold = 0.05
checkpoint_dir_name = "checkpoints"

[training]
training_mode = "examples"
training_format = "light_scratchpad"
skip_overlong_examples = false
curriculum_name = ""

[balanced_validation]
enabled = false
group_by = "kind"
sample_size_per_group = 8
seed = {SEED}
batch_size = ""
'''

config_path = Path(CONFIG_PATH)
config_path.parent.mkdir(parents=True, exist_ok=True)
config_path.write_text(config_text)
print(config_text)
run(["uv", "run", "config", "--size", config_path], cwd=project)


# Generated by notebooks/colab-training.ipynb

[paths]
data_dir = "/content/drive/MyDrive/europa-colab/data/type-place"
output_dir = "/content/drive/MyDrive/europa-colab/runs/type-place-small"

[runtime]
device = "cuda"
seed = 42

[resume]
resume_from = ""
auto_resume = false
additional_epochs = ""

[model]
sequence_length = 64
d_model = 12
n_heads = 3
n_layers = 6
mlp_hidden = 48
dropout = 0.01
position_encoding = "type_place"

[optimization]
batch_size = 512
epochs = 1000
learning_rate = 0.0003
weight_decay = 0.1
grad_clip = 1.0

[logging]
log_interval = 100
eval_batches = 50
exact_match_samples = 256
max_new_tokens = 24

[checkpoint]
checkpoint_keep_last = 5
checkpoint_max_kept = 10
checkpoint_keep_best = 1
checkpoint_jump_threshold = 0.05
checkpoint_dir_name = "checkpoints"

[training]
training_mode = "token_stream"
training_format = "final_only"
skip_overlong_examples = false
curriculum_name = ""

[balanced_validation]
enabled = false
group_by = "kind"
sample_size_per_group = 8
se

In [5]:
# Long-running cell. If Colab disconnects, rerun setup, keep AUTO_RESUME=True, and rerun this.
run(["uv", "run", "train", "train", CONFIG_PATH], cwd=project)



$ uv run train train /content/drive/MyDrive/europa-colab/train-config.toml


KeyboardInterrupt: 

In [ ]:
best_checkpoint = Path(RUN_DIR) / "checkpoint-best.pt"
if not best_checkpoint.exists():
    raise FileNotFoundError(f"No best checkpoint found at {best_checkpoint}")
run(["uv", "run", "evaluate", "--checkpoint", best_checkpoint, "--data-dir", DATA_DIR], cwd=project)


In [ ]:
PROMPT = "<do> <calc> 03000000 + 03000000 ="
run(["uv", "run", "train", "predict", "--checkpoint", best_checkpoint, "--prompt", PROMPT, "--device", DEVICE, "--max-new-tokens", MAX_NEW_TOKENS], cwd=project)
